In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import sys
import warnings

import pandas as pd


# Load configuration
sys.path.append("../")
warnings.filterwarnings("ignore")

from utils import RANDOM_STATE, TEST_SIZE

# Load the data
X = pd.read_csv("../data/train_data.csv", index_col='ID')

# Feature 'CustomerId' est une erreur est doit être éliminé
X.drop(columns='CustomerId', inplace=True)

# Feature 'Surname' n´a aucune signification pour notre projet
X.drop(columns='Surname', inplace=True)

X.head()

,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
ID,,,,,,,,,,,
37765,627,France,Male,28.0,7,131694.04,1,1.0,1.0,161205.61,0
130453,597,France,Male,34.0,2,0.00,2,0.0,1.0,181419.29,0
77297,724,France,Male,39.0,7,0.00,2,1.0,1.0,100862.54,0
40858,663,Germany,Female,56.0,5,118577.24,3,1.0,0.0,61164.45,1
19804,627,France,Female,33.0,5,0.00,2,1.0,1.0,103737.82,0


In [3]:
from sklearn.model_selection import train_test_split
from sklearn import set_config

set_config(transform_output="pandas")

y = X.pop("Exited")
X = X.copy()


X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=TEST_SIZE, random_state=RANDOM_STATE
)

print(f"The X_train set contains: {X_train.shape} elements")
print(f"The y_train set contains: {y_train.shape} elements")
print(f"The X_test set contains: {X_test.shape} elements")
print(f"The y_test set contains: {y_test.shape} elements")

The X_train set contains: (114863, 10) elements
The y_train set contains: (114863,) elements
The X_test set contains: (28716, 10) elements
The y_test set contains: (28716,) elements


## Le modèle

In [4]:
from lightgbm import LGBMClassifier
from sklearn.compose import ColumnTransformer
from sklearn.discriminant_analysis import StandardScaler
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.model_selection import RandomizedSearchCV
from sklearn.pipeline import FeatureUnion, Pipeline
from modelisation.feature_engineering import ChurnCategories, ChurnProduct, ChurnRatios



numerical = ColumnTransformer(
    transformers=[
        ('ratios', ChurnRatios(scale=True), ['Age', 'Tenure', 'Balance', 'EstimatedSalary']),
        ('product', ChurnProduct(scale=True), ['Age', 'Balance', 'NumOfProducts', 'IsActiveMember', 'CreditScore']),
    ],
    verbose_feature_names_out=False,
    remainder='drop'
)


categorical = ColumnTransformer(
    transformers=[
        ('binning', ChurnCategories(encode=True), ['Gender', 'Geography', 'NumOfProducts', 'CreditScore', 'Age', 'Tenure'])
    ],
    verbose_feature_names_out=False,
    remainder='drop'
)

preproc = FeatureUnion(
    transformer_list=[
        ('numerical_pipeline', numerical),
        ('categorical_pipeline', categorical)
    ]
)


gb = GradientBoostingClassifier(random_state=42)

pipe = Pipeline([('prep', preproc),
                 ('model', gb)])

pipe.set_params(**{'model__subsample': 0.8,
 'model__n_estimators': 200,
 'model__min_samples_split': 4,
 'model__min_samples_leaf': 2,
 'model__max_depth': 5,
 'model__learning_rate': 0.05})

# param_grid = {
#     'model__n_estimators': [200,300,400],
#     'model__learning_rate': [0.03,0.05,0.1],
#     'model__max_depth': [3,4,5],
#     'model__subsample': [0.8,1.0],
#     'model__min_samples_split':[2,4],
#     'model__min_samples_leaf':[1,2]
# }
param_grid = {
    'model':[
        GradientBoostingClassifier(
            n_estimators=200,
            random_state=42,
            min_samples_split=4,
            min_samples_leaf=2,
            learning_rate=0.003
            ),
        LGBMClassifier(
        objective='binary',
        random_state=RANDOM_STATE,
        n_jobs=0, 
        is_unbalance=True,  
        learning_rate=0.003, 
        num_leaves=225, 
        max_depth=7, 
        n_estimators=300,
        colsample_bytree=0.9,
        subsample=0.8,
        verbose=0,
        )
    ]
}

search = RandomizedSearchCV(
        pipe, param_grid, n_iter=20,
        scoring='f1', cv=4, random_state=42, n_jobs=-1)
search.fit(X_train, y_train)
best_pipe = search.best_estimator_

[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No f

In [5]:
import numpy as np
from sklearn.metrics import f1_score


proba = best_pipe.predict_proba(X_test)[:,1]
best_t = np.linspace(0.1,0.9,81)[np.argmax([
           f1_score(y_test, (proba>=t).astype(int))
           for t in np.linspace(0.1,0.9,81)])]

In [6]:
search.best_params_

{'model': LGBMClassifier(colsample_bytree=0.9, is_unbalance=True, learning_rate=0.003,
                max_depth=7, n_estimators=300, n_jobs=0, num_leaves=225,
                objective='binary', random_state=0, subsample=0.8, verbose=0)}

In [7]:
best_t

np.float64(0.44000000000000006)

In [8]:
f1_score(y_test, (proba>=best_t))

0.5893310753598645